# Define Catalog and Schema

In [0]:
%sql
USE CATALOG github_lakehouse;
USE SCHEMA silver;

## Create the `issue_comment_events` Silver Table

This step creates the Delta table used to store cleaned and structured **IssueCommentEvent** records in the Silver layer.

The table contains three groups of fields:

- **Common event fields** — Core GitHub event information such as event ID, event timestamp, actor, and repository.
- **Bronze lineage fields** — Source metadata retained from Bronze for data lineage and traceability.
- **IssueCommentEvent-specific fields** — Information about comments associated with GitHub issues or pull requests, including the comment action, issue ID, issue number, comment ID, and target type.

The `comment_target_type` field is a **derived field** used to classify whether the comment belongs to a regular issue or to a pull request conversation.

This separation is important because GitHub represents both issue comments and pull request conversation comments through the `IssueCommentEvent` event type.

The table is stored using **Delta Lake** (`USING DELTA`).

`CREATE TABLE IF NOT EXISTS` ensures that rerunning the notebook does not overwrite an existing table.

> **Note:** This step defines the destination schema only. Classification, cleaning, validation, and loading of IssueCommentEvent records are performed later.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.issue_comment_events (
    -- Common fields
    event_id         STRING,
    event_type       STRING,
    event_timestamp  TIMESTAMP,
    event_date       DATE,
    actor_id         BIGINT,
    actor_login      STRING,
    repo_id          BIGINT,
    repo_name        STRING,

    -- Event-specific fields
    comment_action       STRING,
    issue_id             BIGINT,
    issue_number         BIGINT,
    comment_target_type  STRING,
    comment_id           BIGINT,

    -- Bronze lineage
    _source_file     STRING,
    _source_date     DATE,
    _ingested_at     TIMESTAMP
)
USING DELTA;